# Qwen Multimodal Chat on Colab

Qwen3.8-27B（チャット・画像理解）＋ Qwen-Image-2.1（画像生成・編集）を、ブラウザからChatGPTのように使うための起動用Notebookです。

1. ［ランタイム］→［ランタイムのタイプを変更］で **A100**（推奨）または **L4** を選ぶ
2. 左の🔑 **Secrets** に `HF_TOKEN` を登録（任意だけど推奨。ダウンロードが速く安定します）
3. 上から順に4つのセルを実行 → 最後に出るリンクからチャット画面を開く

ロジックはすべて `src/qmc/` にあり、このNotebookは起動するだけです。詳しくは README を見てください。

In [ ]:
# Cell 1: リポジトリを取得
import os
import subprocess

REPO = "https://github.com/moruku36/qwen-multimodal-colab"
if not os.path.exists("/content/qwen-multimodal-colab"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/qwen-multimodal-colab"], check=True)
else:
    subprocess.run(["git", "-C", "/content/qwen-multimodal-colab", "pull", "--ff-only"], check=True)
%cd /content/qwen-multimodal-colab
!git log --oneline -1

In [ ]:
# Cell 2: 依存関係（Colab標準のtorch/CUDAはそのまま）+ llama.cpp（2回目以降はDriveキャッシュから数秒）
%pip install -q -r requirements-colab.txt
import sys

sys.path.insert(
    0, "/content/qwen-multimodal-colab/src"
)  # editable install is not visible in a running kernel
from qmc import colab

colab.setup()  # Drive マウント（llama.cpp のキャッシュ先にも使う）
colab.install_llama_cpp()

In [ ]:
# Cell 3: 設定（必要なら変更）
SHARE = False  # True にすると *.gradio.live の公開URLが発行されます（誰でもアクセス可能！）
PREFETCH = False  # True: 先にモデルを全部ダウンロード（初回 約60GB）
PROFILE = None  # None=自動判定 / "a100_80" / "a100_40" / "l4"

if PREFETCH:
    colab.prefetch_models()
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
# Cell 4: 起動（表示されたリンクからチャット画面を開く）
app = colab.launch(share=SHARE, profile=PROFILE)

### メモ
- 初回はモデルのダウンロード（27B GGUF 約20GB + Qwen-Image-2.1 約40GB）に時間がかかります。
- **L4** では Chat と画像モデルを入れ替えながら動きます（切替時に数十秒〜1分ほど待ちます）。
- 会話と画像は Google Drive の `MyDrive/qwen-multimodal-colab/data/` に保存され、再起動後も復元されます。
- 停止するには［ランタイム］→［セッションを終了］。